### PREREQUISITES

#### Importing Modules and Tables

In [200]:
import pandas as pd
import numpy as np
import re

In [265]:


print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 1.26.4
Pandas version: 2.2.2


In [201]:
candidate_df = pd.read_csv(
    r"C:\Users\AshutoshMishra\Downloads\CTGD2025\CTGD2025\candidates_final.csv",
    encoding='latin1'
)
vacancy_df = pd.read_csv(r"C:\Users\AshutoshMishra\Downloads\CTGD2025\CTGD2025\vacancy_table_raw.csv")

In [202]:
candidate_df['dob'] = pd.to_datetime(candidate_df['dob'])

In [203]:
import pandas as pd
import numpy as np


CANDIDATE_REQUIRED_COLS = [
    "regno", "rollno", "cand_name", "dob",
    "cat1", "cat2", "cat3", "gender",
    "statecode_considered", "naxal_district", "border_district",
    "parta_gi", "partb_ga", "normalized_score", "total_marks", "ncc_marks",
    "rejection_provision", "post_pref", "agerelax_code",
    "exs_reservation", "service_period",
    "ht_rlx_code", "chst_rlx_code", "height_relax", "chest_relax", "height_chest_relax",
]

VALID_CAT1       = {0, 1, 2, 6, 9}
VALID_CAT2       = {3}
VALID_GENDER     = {1, 2, 3}
VALID_AGERELAX   = {1, 2, 3, 4, 5, 6}


def _header(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")


def check_candidate_df(df):

    _header("CANDIDATE_DF SANITY CHECK")
    failures = 0

    print("\n[1] Required Columns")
    missing_cols = [c for c in CANDIDATE_REQUIRED_COLS if c not in df.columns]
    if missing_cols:
        print(f"    FAIL - Missing columns: {missing_cols}")
        failures += 1
    else:
        print(f"    All {len(CANDIDATE_REQUIRED_COLS)} required columns present")

    print("\n[2] Row Count")
    print(f"    Total rows: {len(df):,}")

    print("\n[3] Duplicate Checks")
    dup_rollno = df['rollno'].duplicated().sum()
    dup_regno  = df['regno'].duplicated().sum()
    if dup_rollno:
        print(f"    FAIL - Duplicate rollno found: {dup_rollno} rows")
        failures += 1
    else:
        print("    No duplicate rollno")
    if dup_regno:
        print(f"    FAIL - Duplicate regno found: {dup_regno} rows")
        failures += 1
    else:
        print("    No duplicate regno")

    print("\n[4] DOB Checks")
    if not pd.api.types.is_datetime64_any_dtype(df['dob']):
        print("    FAIL - dob column is NOT datetime — run pd.to_datetime(candidate_df['dob']) first")
        failures += 1
    else:
        print("    dob column is datetime type")
        null_dob = df['dob'].isnull().sum()
        if null_dob:
            print(f"    WARN - {null_dob} rows have NULL dob")
        else:
            print("    No NULL dob values")

    print("\n[5] Gender Check")
    invalid_gender = df[~df['gender'].isin(VALID_GENDER)]
    if len(invalid_gender):
        print(f"    FAIL - {len(invalid_gender)} rows with invalid gender (expected {VALID_GENDER})")
        print(f"    Values found: {df['gender'].value_counts().to_dict()}")
        failures += 1
    else:
        print(f"    All gender values valid: {df['gender'].value_counts().to_dict()}")

    print("\n[6] cat1 (Category) Check")
    try:
        cat1_vals = df['cat1'].dropna().astype(float).astype(int)
        invalid_cat1 = cat1_vals[~cat1_vals.isin(VALID_CAT1)]
        if len(invalid_cat1):
            print(f"    FAIL - {len(invalid_cat1)} rows with invalid cat1 (expected {VALID_CAT1})")
            print(f"    Values found: {invalid_cat1.value_counts().to_dict()}")
            failures += 1
        else:
            print(f"    All cat1 values valid. Distribution: {cat1_vals.value_counts().to_dict()}")
        null_cat1 = df['cat1'].isnull().sum()
        if null_cat1:
            print(f"    WARN - {null_cat1} NULL cat1 values")
    except Exception as e:
        print(f"    WARN - Could not validate cat1: {e}")

    print("\n[7] cat2 Check (should be 3 or NULL)")
    try:
        cat2_nonnull = df['cat2'].dropna()
        invalid_cat2 = cat2_nonnull[~cat2_nonnull.astype(float).astype(int).isin(VALID_CAT2)]
        if len(invalid_cat2):
            print(f"    FAIL - {len(invalid_cat2)} rows with unexpected cat2 values: {invalid_cat2.unique()[:10]}")
            failures += 1
        else:
            print(f"    cat2 values OK — {(df['cat2'].dropna().astype(float).astype(int) == 3).sum()} candidates have cat2=3 (OBC)")
    except Exception as e:
        print(f"    WARN - Could not validate cat2: {e}")

    print("\n[8] agerelax_code Check")
    arc_series = df['agerelax_code'].dropna()
    try:
        arc_int = arc_series.astype(float).astype(int)
        invalid_arc = arc_int[~arc_int.isin(VALID_AGERELAX)]
        if len(invalid_arc):
            print(f"    FAIL - {len(invalid_arc)} rows with invalid agerelax_code: {invalid_arc.unique()[:10]}")
            failures += 1
        else:
            print(f"    agerelax_code values OK. Distribution: {arc_int.value_counts().to_dict()}")
            print(f"    NULL agerelax_code: {df['agerelax_code'].isnull().sum()} rows")
    except Exception as e:
        print(f"    WARN - Could not validate agerelax_code: {e}")

    print("\n[12] Naxal / Border District Flags")
    for col in ['naxal_district', 'border_district']:
        if col in df.columns:
            unique_vals = set(df[col].dropna().unique())
            valid_sets = [{"Yes", "No"}, {True, False}, {0, 1}]
            if not any(unique_vals <= v for v in valid_sets):
                print(f"    FAIL - {col} has unexpected values: {unique_vals}")
                failures += 1
            else:
                print(f"    {col} values OK: {df[col].value_counts().to_dict()}")

    print("\n[14] post_pref Check")
    null_post  = df['post_pref'].isnull().sum()
    empty_post = (df['post_pref'].astype(str).str.strip() == '').sum()
    if null_post or empty_post:
        print(f"    WARN - {null_post} NULL + {empty_post} empty post_pref entries")
    else:
        print("    post_pref has no NULL/empty values")

    print("\n[16] rejection_provision Check")
    if 'rejection_provision' in df.columns:
        rejected = (df['rejection_provision'].notnull() &
                    (df['rejection_provision'].astype(str).str.strip() != '')).sum()
        print(f"    rejection_provision — {rejected} candidates have a rejection flag")
        print(f"    Distribution: {df['rejection_provision'].value_counts(dropna=False).to_dict()}")

    print("\n[18] service_period Format Check (for ARC code 3)")
    arc3_mask = df['agerelax_code'].dropna().astype(float).astype(int) == 3
    arc3_index = df['agerelax_code'].dropna().astype(float).astype(int)[arc3_mask].index
    arc3_candidates = df.loc[arc3_index]
    if len(arc3_candidates):
        bad_sp    = arc3_candidates['service_period'].isna().sum()
        pattern   = r'\d+ Year[s]* \d+ Month[s]* \d+ Day[s]*'
        valid_sp  = arc3_candidates['service_period'].dropna().str.match(pattern).sum()
        total_sp  = arc3_candidates['service_period'].dropna().shape[0]
        if bad_sp:
            print(f"    FAIL - {bad_sp} ex-servicemen (ARC=3) have NULL service_period")
            failures += 1
        elif valid_sp < total_sp:
            print(f"    WARN - {total_sp - valid_sp} service_period entries do not match 'X Years Y Months Z Days' format")
        else:
            print(f"    All {len(arc3_candidates)} ex-servicemen have valid service_period format")
    else:
        print("    No ARC code 3 (ex-servicemen) candidates in dataset")

    _header("CANDIDATE_DF SUMMARY")
    if failures == 0:
        print("    All checks passed. Dataset is ready for processing.")
    else:
        print(f"    {failures} CRITICAL issue(s) found. Fix before proceeding!")

    return failures == 0


VACANCY_REQUIRED_COLS = [
    "state_code", "gender", "post_code", "area", "category_code",
    "initial", "current", "allocated", "allocated_hc",
]


def check_vacancy_df(df):

    _header("VACANCY_DF SANITY CHECK")
    failures = 0

    print("\n[1] Required Columns")
    missing_cols = [c for c in VACANCY_REQUIRED_COLS if c not in df.columns]
    present_cols = [c for c in VACANCY_REQUIRED_COLS if c in df.columns]
    if missing_cols:
        print(f"    FAIL - Missing columns: {missing_cols}")
        failures += 1
    else:
        print(f"    All {len(VACANCY_REQUIRED_COLS)} required columns present:")
        print(f"    {present_cols}")

    print("\n[7] Vacancy Counts Non-Negative")
    for col in ['initial', 'current', 'allocated', 'allocated_hc']:
        if col in df.columns:
            neg = (df[col] < 0).sum()
            if neg:
                print(f"    FAIL - {neg} negative values in '{col}'")
                failures += 1
            else:
                print(f"    '{col}' — no negative values (total={df[col].sum():,})")

    print("\n[8] current <= initial Check")
    if 'initial' in df.columns and 'current' in df.columns:
        over = (df['current'] > df['initial']).sum()
        if over:
            print(f"    FAIL - {over} rows where current > initial")
            failures += 1
        else:
            print("    current <= initial for all rows")

    print("\n[9] allocated_hc <= initial Check")
    if 'initial' in df.columns and 'allocated_hc' in df.columns:
        over_hc = (df['allocated_hc'] > df['initial']).sum()
        if over_hc:
            print(f"    FAIL - {over_hc} rows where allocated_hc > initial")
            failures += 1
        else:
            print("    allocated_hc <= initial for all rows")

    _header("VACANCY_DF SUMMARY")
    if failures == 0:
        print("    All checks passed. Vacancy data is ready.")
    else:
        print(f"    {failures} CRITICAL issue(s) found. Fix before proceeding!")

    return failures == 0


CUTOFF_REQUIRED_COLS    = ["serial_id", "category", "total"]
EXPECTED_CUTOFF_CATS    = {0, 1, 2, 3, 6, 9}
CUTOFF_CAT_NAMES        = {0: "UR", 1: "SC", 2: "ST", 3: "OBC/EXS", 6: "EWS", 9: "Migrated/UR"}


def check_cutoff_df(df):

    _header("CUTOFF_DF SANITY CHECK")
    failures = 0

    print("\n[1] Required Columns")
    missing_cols = [c for c in CUTOFF_REQUIRED_COLS if c not in df.columns]
    present_cols = [c for c in df.columns]
    if missing_cols:
        print(f"    FAIL - Missing columns: {missing_cols}")
        failures += 1
    else:
        print(f"    All required columns present:")
        print(f"    {present_cols}")

    print("\n[2] Expected Categories Present")
    if 'category' in df.columns:
        present_cats = set(df['category'].astype(int).unique())
        missing_cats = EXPECTED_CUTOFF_CATS - present_cats
        extra_cats   = present_cats - EXPECTED_CUTOFF_CATS
        if missing_cats:
            print(f"    FAIL - Missing categories: {missing_cats}")
            failures += 1
        else:
            print(f"    All expected categories present: {sorted(present_cats)}")
        if extra_cats:
            print(f"    WARN - Extra/unexpected categories found: {extra_cats}")

    print("\n[3] Duplicate Category Check")
    if 'category' in df.columns:
        dup_cats = df['category'].duplicated().sum()
        if dup_cats:
            print(f"    FAIL - {dup_cats} duplicate category entries")
            failures += 1
        else:
            print("    No duplicate categories")

    print("\n[5] Cutoff Table (for manual verification)")
    if 'category' in df.columns and 'total' in df.columns:
        display_df = df[['category', 'total']].copy()
        display_df['category_name'] = display_df['category'].map(CUTOFF_CAT_NAMES).fillna("Unknown")
        print(display_df[['category', 'category_name', 'total']].to_string(index=False))

    _header("CUTOFF_DF SUMMARY")
    if failures == 0:
        print("    All checks passed. Cutoff table is ready.")
    else:
        print(f"    {failures} CRITICAL issue(s) found. Fix before proceeding!")

    return failures == 0

In [204]:



run_all_sanity_checks(candidate_df,vacancy_df,cutoff_df)



  CTGD 2025 — MASTER SANITY CHECK


  CANDIDATE_DF SANITY CHECK

[1] Required Columns
    All 26 required columns present

[2] Row Count
    Total rows: 73,465

[3] Duplicate Checks
    No duplicate rollno
    No duplicate regno

[4] DOB Checks
    dob column is datetime type
    No NULL dob values

[5] Gender Check
    All gender values valid: {2: 66126, 1: 7339}

[6] cat1 (Category) Check
    All cat1 values valid. Distribution: {6: 30730, 1: 14298, 2: 10448, 0: 9051, 9: 8938}

[7] cat2 Check (should be 3 or NULL)
    cat2 values OK — 288 candidates have cat2=3 (OBC)

[8] agerelax_code Check
    agerelax_code values OK. Distribution: {2: 30623, 1: 24733, 3: 288, 5: 2, 6: 1, 4: 1}
    NULL agerelax_code: 17817 rows

[12] Naxal / Border District Flags
    naxal_district values OK: {False: 51729, True: 21736}
    border_district values OK: {False: 49218, True: 24247}

[14] post_pref Check
    post_pref has no NULL/empty values

[16] rejection_provision Check
    rejection_provision — 

True

###### ARC Table from Notice

In [205]:
arc_dict = [
    {'arc_code': 1, 'arc_year': 5, 'arc_cat1': '1,2', 'arc_cat2': None, 'arc_gender': '1,2,3', 'arc_year_against_ur': None},
    {'arc_code': 2, 'arc_year': 3, 'arc_cat1': '6', 'arc_cat2': None, 'arc_gender': '1,2,3', 'arc_year_against_ur': None},
    {'arc_code': 3, 'arc_year': 3, 'arc_cat1': '0,1,2,6,9', 'arc_cat2': '3', 'arc_gender': '1,2,3', 'arc_year_against_ur': None},
    {'arc_code': 4, 'arc_year': 5, 'arc_cat1': '0,9', 'arc_cat2': None, 'arc_gender': '1,2,3', 'arc_year_against_ur': None},
    {'arc_code': 5, 'arc_year': 8, 'arc_cat1': '6', 'arc_cat2': None, 'arc_gender': '1,2,3', 'arc_year_against_ur': 5},
    {'arc_code': 6, 'arc_year': 10, 'arc_cat1': '1,2', 'arc_cat2': None, 'arc_gender': '1,2,3', 'arc_year_against_ur': 5},
]

arc_df = pd.DataFrame(arc_dict)

#### Cutoff Table from Notice

In [206]:
cutoff_data = [
    {'serial_id': 1, 'category': 0, 'total': 40},
    {'serial_id': 2, 'category': 1, 'total': 32},
    {'serial_id': 3, 'category': 2, 'total': 32},
    {'serial_id': 4, 'category': 3, 'total': 32},
    {'serial_id': 5, 'category': 6, 'total': 40},
    {'serial_id': 6, 'category': 9, 'total': 48},
]

cutoff_df = pd.DataFrame(cutoff_data)

### MERIT

#### CODE

In [207]:
def get_merit(candidate_df):    
    
    candidate_df['merit'] = None
    
    candidates =  candidate_df[
    (candidate_df['normalized_score'].notnull())
    ].sort_values(by = ['total_marks', 'parta_gi', 'partb_ga', 'dob', 'cand_name'], ascending=[False, False, False, True, True])
    
    candidates['merit'] = range(1, len(candidates) + 1)

    rollno_to_merit = candidates.set_index('rollno')['merit'].to_dict()
    candidate_df['merit'] = candidate_df['rollno'].map(rollno_to_merit)
    
    return "{flag_name} updated successfully for {merit} candidates".format(
        flag_name='merit',
        merit=len(candidates))

#### EXECUTION

In [208]:
get_merit(candidate_df)

'merit updated successfully for 73465 candidates'

### CUTOFF FLAG

#### CODE

In [209]:
def get_cutoff(cutoff_df, candidate_df):

    cutoff = 'cutoff_flag'
    sorted_candidates = candidate_df[candidate_df['merit'].notnull()].sort_values(by='merit')

    candidate_df[cutoff] = None
    cutoff_dict = {}
    
    for index, row in cutoff_df.iterrows():
        category = int(row['category'])
        total = float(row['total'])
        cutoff_dict[category] = total
    
    
    def calculate_cutoff(candidate):
        cutoff = ''
    
        if (candidate['normalized_score'] >= cutoff_dict[9]):
            cutoff += '9'

        for cat in [0, 1, 2, 6]:
            if (candidate['cat1'] == cat) & (cat in cutoff_dict):
                if (candidate['normalized_score'] >= cutoff_dict[cat]):
                    cutoff += str(cat)

        if (candidate['cat2'] == 3) & (3 in cutoff_dict):
            if (candidate['normalized_score'] >= cutoff_dict[3]):
                cutoff += '3'

        return cutoff

    sorted_candidates[cutoff] = sorted_candidates.apply(lambda row: calculate_cutoff(row), axis=1)
    candidate_df.update(sorted_candidates)

    return f"{cutoff} updated successfully"

#### EXECUTION

In [210]:
get_cutoff(cutoff_df, candidate_df)

'cutoff_flag updated successfully'

In [211]:
candidate_df[['rollno', 'cutoff_flag']].head(5)

,rollno,cutoff_flag
0,5112003442,90
1,5105087686,96
2,1401048202,96
3,8604012588,91
4,3205039907,96


### DOB FLAG

#### PREREQIUSITES

In [212]:
mask = candidate_df['cat1'].isnull()
candidate_df.loc[mask, 'cat1'] = pd.NA
candidate_df.loc[~mask, 'cat1'] = candidate_df.loc[~mask, 'cat1'].astype('Int64').astype(str)

mask = candidate_df['cat2'].isnull()
candidate_df.loc[mask, 'cat2'] = pd.NA
candidate_df.loc[~mask, 'cat2'] = candidate_df.loc[~mask, 'cat2'].astype('Int64').astype(str)

mask = candidate_df['agerelax_code'].isnull()
candidate_df.loc[mask, 'agerelax_code'] = ''
candidate_df.loc[~mask, 'agerelax_code'] = candidate_df.loc[~mask, 'agerelax_code'].astype(int).astype(str)

C:\Users\AshutoshMishra\AppData\Local\Temp\ipykernel_21228\3681777594.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['0' '6' '6' ... '6' '1' '2']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_df.loc[~mask, 'cat1'] = candidate_df.loc[~mask, 'cat1'].astype('Int64').astype(str)
C:\Users\AshutoshMishra\AppData\Local\Temp\ipykernel_21228\3681777594.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3'
 '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3'
 '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3'
 '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3'
 '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3' '3'
 '3' '3' 

In [213]:
candidate_df[['exsm_yrs', 'exsm_months', 'exsm_days']] = candidate_df['service_period'].str.extract(r'(\d+) Year[s]* (\d+) Month[s]* (\d+) Day[s]*')

candidate_df[['exsm_yrs', 'exsm_months', 'exsm_days']] = candidate_df[['exsm_yrs', 'exsm_months', 'exsm_days']].apply(pd.to_numeric)

#### CODE

In [214]:
def get_dob_flag(candidate_df, arc_df):
    
    dob_flag = 'dob_flag'
    dob_to_date = pd.to_datetime('01/02/2007', format="%m/%d/%Y")
    dob_from_date = pd.to_datetime('01/01/2002', format="%m/%d/%Y")
    
    arc_year_dict = dict(zip(arc_df['arc_code'].astype(str), arc_df['arc_year']))

    candidate_df[dob_flag] = ''

    def calculate_verification_flag(row):
        dob = row['dob']
        verification_flag = ''
        
        if dob >= dob_to_date:
            verification_flag = 'U'
        
        elif dob_from_date < dob < dob_to_date:
            verification_flag = '9'
        
        elif dob <= dob_from_date:
            agerelax_code = row['agerelax_code']
            
            if pd.isna(agerelax_code) or agerelax_code == '':
                agerelax_code = '99'
            
            if isinstance(agerelax_code, str) and len(agerelax_code.strip()) == 1:
                agerelax_code = agerelax_code.strip()
            
            age_rlx_year = arc_year_dict.get(agerelax_code, 0)
            
            if agerelax_code == '3' :
                ex_service_years = row['exsm_yrs']
                ex_service_months = row['exsm_months']
                ex_service_days = row['exsm_days']
                age_rlx_year += int(ex_service_years) if ex_service_years else 0
                new_dob = dob + pd.DateOffset(years=age_rlx_year, months=ex_service_months, days=ex_service_days)
            
            else:
                extract_year = dob.year
                extract_month = dob.month
                extract_day = dob.day
                new_year = extract_year + int(age_rlx_year)
                
                if extract_month == 2 and extract_day == 29:
                    extract_day = 28
            
                new_dob = pd.to_datetime(f"{new_year}-{extract_month}-{extract_day}", format='%Y-%m-%d')
                
            if new_dob > dob_from_date:
                if len(agerelax_code) < 2:
                    verification_flag = '0' + agerelax_code
                else:
                    verification_flag = agerelax_code
            else:
                verification_flag = '99'

        return verification_flag

    sorted_candidates = candidate_df[(candidate_df['merit'].notnull())]
    
    sorted_candidates[dob_flag] = sorted_candidates.apply(calculate_verification_flag, axis=1)

    candidate_df.update(sorted_candidates)
    
    return f"{dob_flag} updated successfully"

#### EXECUTION

In [215]:
get_dob_flag(candidate_df, arc_df)

'dob_flag updated successfully'

In [216]:
candidate_df[['rollno', 'dob_flag']].head()

,rollno,dob_flag
0,5112003442,9
1,5105087686,02
2,1401048202,02
3,8604012588,9
4,3205039907,9


In [217]:
candidate_df['dob_flag'].unique()

array(['9', '02', '01', '03', '99', '05'], dtype=object)

In [218]:
null_cutoff_rows = candidate_df[
    candidate_df['dob'] == '2002-01-01'
][['dob', 'cat1', "cat2", "cat3","merit","agerelax_code","dob_flag"]].head(100)


print(null_cutoff_rows)


             dob cat1 cat2  cat3  merit agerelax_code dob_flag
380   2002-01-01    6  NaN   NaN   2655             2       02
893   2002-01-01    6  NaN   NaN   2157             2       02
1479  2002-01-01    6  NaN   NaN  38940             2       02
1937  2002-01-01    6  NaN   NaN  64565             2       02
2027  2002-01-01    2  NaN   NaN  42036             1       01
...          ...  ...  ...   ...    ...           ...      ...
36147 2002-01-01    1  NaN   NaN  22649             1       01
36197 2002-01-01    6  NaN   NaN  11641             2       02
36903 2002-01-01    6  NaN   NaN  17327             2       02
38372 2002-01-01    6  NaN   NaN   7686             2       02
38521 2002-01-01    6  NaN   NaN  57687             2       02

[100 rows x 7 columns]


### CATSEL DOB FLAG

#### PREREQUISITES

In [219]:
arc_df['arc_code'] = arc_df['arc_code'].astype(str)

mask = arc_df['arc_cat2'].isnull()
arc_df.loc[mask, 'arc_cat2'] = ''

In [220]:
arc_df['arc_code'] = arc_df['arc_code'].apply(lambda x: str(x).zfill(2))

#### CODE

In [221]:
def get_catsel_dob(candidate_df, arc_df):
    
    dob_flag = 'dob_flag'
    catsel_dob_flag = 'catsel_dob_flag'
    dob_from_date = pd.to_datetime('01/01/2002', format="%m/%d/%Y")

    candidate_df[catsel_dob_flag] = ''
    
    filtered_and_sorted_candidates = candidate_df[(candidate_df['merit'].notnull()) 
    & (candidate_df[dob_flag].notnull()) 
    & (candidate_df[dob_flag] != '')]
    
    def calculate_catsel_dob(row, dob_flag_name):
        dob_flag = row[dob_flag_name]
       
        if dob_flag is not None and dob_flag != '' and dob_flag != '9':
            matching_rows = arc_df[arc_df['arc_code'] == dob_flag]
            if not matching_rows.empty:
                catsel = matching_rows.iloc[0]
                catsel_cat1 = catsel['arc_cat1'].split(',')
                catsel_cat2 = catsel['arc_cat2'].split(',')
                catsel_gender = catsel['arc_gender'].split(',')
                condition = (str(row['cat1']) in catsel_cat1) & (str(int(row['gender'])) in catsel_gender)
                catsel_dob = ''
                if row['cat2'] == '3' and row['cat2'] in catsel_cat2:
                    catsel_dob = row['cat2']
                else:
                    if pd.isnull(catsel['arc_year_against_ur']):
                        catsel['arc_year_against_ur'] = 0
                    dob_datetime = pd.to_datetime(row['dob'])
                    extract_year = dob_datetime.year
                    extract_month = dob_datetime.month
                    extract_day = dob_datetime.day
                    if (extract_day == 29) and (extract_month == 2):
                        extract_day = 28
                    new_year = extract_year + int(catsel['arc_year_against_ur'])
                    new_dob = pd.to_datetime(f'{new_year}-{extract_month}-{extract_day}')
                    catsel_dob_decision = new_dob > dob_from_date
                    
                    catsel_dob = '9' if catsel_dob_decision else row['cat1']
                    
                return catsel_dob if condition else ''
            else:
                return ''
        else:
            return dob_flag
    
    filtered_and_sorted_candidates[catsel_dob_flag] = filtered_and_sorted_candidates.apply(calculate_catsel_dob, args = (dob_flag, ), axis=1)
    candidate_df.update(filtered_and_sorted_candidates)
    
    print('Successfully updated ' + catsel_dob_flag)

#### EXECUTION

In [222]:
get_catsel_dob(candidate_df, arc_df)

Successfully updated catsel_dob_flag


In [223]:
candidate_df[['rollno', 'catsel_dob_flag']].head()

,rollno,catsel_dob_flag
0,5112003442,9
1,5105087686,6
2,1401048202,6
3,8604012588,9
4,3205039907,9


In [224]:
candidate_df['catsel_dob_flag'].unique()

array(['9', '6', '2', '1', '3', ''], dtype=object)

### CATSEL FLAG

In [225]:
mask = candidate_df['catsel_dob_flag'].isnull()
candidate_df.loc[mask, 'catsel_dob_flag'] = ''
candidate_df.loc[~mask, 'catsel_dob_flag'] = candidate_df.loc[~mask, 'catsel_dob_flag'].astype(str)

#### CODE

In [226]:
def get_catsel(candidate_df):

    cutoff_flag = 'cutoff_flag'
    catsel_flag = 'catsel'
    catsel_dob_flag = 'catsel_dob_flag'
    
    candidate_df[catsel_flag] = ''
    
    candidates = candidate_df[(candidate_df['merit'].notnull())].sort_values(by='rollno')
    
    def update_cutoff_flag(candidate_record):
        if candidate_record['cat2'] == '3' and candidate_record['exs_reservation'] == "No":
            candidate_record[cutoff_flag] = candidate_record[cutoff_flag].replace('3', '').strip()

        if candidate_record[catsel_dob_flag] in ['1', '2', '6', '4', '5', '7', '8']:
            return candidate_record[cutoff_flag].replace('9', '').strip()
        elif pd.isna(candidate_record[catsel_dob_flag]) or candidate_record[catsel_dob_flag] == '':
            return ''
        else:
            return candidate_record[cutoff_flag]
    
    candidate_df[catsel_flag] = candidates.apply(update_cutoff_flag, axis=1)

    return 'Successfully updated '+ catsel_flag

#### EXECUTION

In [227]:
get_catsel(candidate_df)

'Successfully updated catsel'

In [228]:
candidate_df[['rollno', 'catsel']].head()

,rollno,catsel
0,5112003442,90
1,5105087686,6
2,1401048202,6
3,8604012588,91
4,3205039907,96


In [229]:
candidate_df['catsel'].unique()

array(['90', '6', '91', '96', '2', '92', '1', '9', '0', '', '3', '963',
       '93', '63', '13', '913', '903', '23'], dtype=object)

In [230]:
candidate_df['catsel_backup'] = candidate_df['catsel']

In [231]:
candidate_df['catsel_backup'].unique()

array(['90', '6', '91', '96', '2', '92', '1', '9', '0', '', '3', '963',
       '93', '63', '13', '913', '903', '23'], dtype=object)

In [232]:
candidate_df.columns

Index(['regno', 'rollno', 'cand_name', 'dob', 'cat1', 'cat2', 'cat3', 'gender',
       'statecode_considered', 'naxal_district', 'border_district', 'parta_gi',
       'partb_ga', 'normalized_score', 'total_marks', 'rejection_provision',
       'post_pref', 'agerelax_code', 'ncc_marks', 'exs_reservation',
       'service_period', 'ht_rlx_code', 'chst_rlx_code', 'height_relax',
       'chest_relax', 'height_chest_relax', 'merit', 'cutoff_flag', 'exsm_yrs',
       'exsm_months', 'exsm_days', 'dob_flag', 'catsel_dob_flag', 'catsel',
       'catsel_backup'],
      dtype='object')

In [233]:
catsel = 'catsel'   # ← apne actual column ka naam yahan likho

mask = candidate_df['height_chest_relax'] == "Yes"

candidate_df.loc[mask, catsel] = (
    candidate_df.loc[mask, catsel]
    .astype(str)
    .str.replace('9', '', regex=False)
    .str.strip()
)



In [234]:
null_cutoff_rows = candidate_df[
    candidate_df['height_chest_relax'] == 'Yes'
][['catsel', 'catsel_backup']].head(10)


print(null_cutoff_rows)


    catsel catsel_backup
8        2            92
25       2             2
128      2             2
133      2            92
156      2             2
170      2             2
181      2             2
206      2            92
267      2            92
282      2            92


In [235]:
candidate_df[candidate_df['catsel'] == ''].shape[0]

65

## ALLOCATION

In [236]:
cand = candidate_df.copy()
vac = vacancy_df.copy()

In [237]:
vac['allocated'] = 0

In [238]:
cand[['allocated_category', 'allocated_post', 'allocated_state', 'allocated_area', 'allocated_against_ur']] = None

In [239]:
mask = cand['post_pref'].isnull()
cand.loc[mask, 'post_pref'] = ''
cand.loc[~mask, 'post_pref'] = cand.loc[~mask, 'post_pref'].astype(str)

mask = cand['catsel'].isnull()
cand.loc[mask, 'catsel'] = ''
cand.loc[~mask, 'catsel'] = cand.loc[~mask, 'catsel'].astype(str)

In [240]:
vac['allocated_hc'] = 0
vac['allocated_hc_prev'] = 0

In [241]:
vac['min_marks_cand_dob_prev'] = pd.to_datetime(vac['min_marks_cand_dob_prev'])

In [242]:
vac['left_vacancy'] = vac['current']
vacancy_dict = {}
vac['key'] = vac['state_code'].astype(str) + vac['gender'].astype(str) + vac['post_code'].astype(str) + vac['area'].astype(str) + vac['category_code'].astype(str)
for index, row in vac.iterrows():
    key = row['key']
    vacancy_dict[key] = row.to_dict()
    vacancy_dict[key]['min_marks_prev'] = row.get('min_marks_prev', 0)
    vacancy_dict[key]['min_marks_parta_prev'] = row.get('min_marks_parta_prev', 0)
    vacancy_dict[key]['min_marks_partb_prev'] = row.get('min_marks_partb_prev', 0)
    vacancy_dict[key]['min_marks_cand_dob_prev'] = row.get('min_marks_cand_dob_prev', pd.Timestamp.min)

#### CODE

In [243]:
def update_allocation(candidates_df, state, gender, post, area, category, cat1, allocated_against_ur, candidate, vacancy_dict):
    if str(category) in ['3', '4', '5', '7', '8']:
    
        key_cat1 = str(state) + str(gender) + str(post) + str(area) + str(cat1)
        
        if (key_cat1 not in vacancy_dict.keys()) or (vacancy_dict[key_cat1]['initial'] == 0):
            keyCat2= str(state) + str(gender) + str(post) + str(area) + '9'
            
            if (keyCat2 not in vacancy_dict.keys()) or (vacancy_dict[keyCat2]['initial'] == 0):
                return False
            else:
                if vacancy_dict[keyCat2]['allocated_hc'] != vacancy_dict[keyCat2]['initial']:
                    vacancy_dict[keyCat2]['allocated_hc'] += 1
                allocated_against_ur = '1'
        else:
            if vacancy_dict[key_cat1]['allocated_hc'] != vacancy_dict[key_cat1]['initial']:
                vacancy_dict[key_cat1]['allocated_hc'] += 1
            
            else :
                return False
                        
    candidates_df.loc[candidate.name, 'allocated_category'] = category
    candidates_df.loc[candidate.name, 'allocated_state'] = state
    candidates_df.loc[candidate.name, 'allocated_area'] = area
    candidates_df.loc[candidate.name, 'allocated_against_ur'] = allocated_against_ur
    candidates_df.loc[candidate.name, 'allocated_post'] = post
    
    
    return True

In [244]:
def allocate_candidates(candidates_df, vacancy_dict):
    filtered_candidates = candidates_df[(candidates_df['merit'].notnull()) | (candidates_df['catsel'] != '')].sort_values(by='merit')
 
    for idx, candidate in filtered_candidates.iterrows():
        allocated = False
        roll = candidate['rollno']
        post_preference = candidate['post_pref']
        DOB = candidate['dob']
        gender = candidate['gender']
        state = int(candidate['statecode_considered'])
        total_marks = candidate['total_marks']
        part_a = candidate['parta_gi']
        part_b = candidate['partb_ga']
        cat1 = str(candidate['cat1'])
        
        for post in post_preference.split(','):
            catsel = candidate['catsel']
 
            for category in catsel:
                allocated_against_ur = ''
 
                # Post preference "H" logic
                if str(post) in ['H','G']:
                    key = str(39) + str(gender) + str(post) + "G" + str(category)
                    if key in vacancy_dict and vacancy_dict[key]['current'] > 0:
                        min_marks_prev = vacancy_dict[key].get('min_marks_prev', 0)
                        min_marks_parta_prev = vacancy_dict[key].get('min_marks_parta_prev', 0)
                        min_marks_partb_prev = vacancy_dict[key].get('min_marks_partb_prev', 0)
                        min_marks_cand_dob_prev = vacancy_dict[key].get('min_marks_cand_dob_prev', pd.Timestamp.min)
 
                        allocate = False
                        if total_marks > min_marks_prev:
                            allocate = True
                        elif total_marks == min_marks_prev:
                            if part_a > min_marks_parta_prev:
                                allocate = True
                            elif part_a == min_marks_parta_prev:
                                if part_b > min_marks_partb_prev:
                                    allocate = True
                                elif part_b == min_marks_partb_prev:
                                    if DOB < min_marks_cand_dob_prev:
                                        allocate = True
                                    elif DOB == min_marks_cand_dob_prev:
                                        allocate = True
 
                        if allocate:
                            post = vacancy_dict[key]['post_code']
                            allocated = update_allocation(candidates_df, 39, gender, post, "G", category, cat1, allocated_against_ur, candidate, vacancy_dict)
                            if allocated:
                                vacancy_dict[key]['current'] -= 1
                                vacancy_dict[key]['allocated'] += 1
                                break
 
                # Naxal-affected district logic
                if candidate['naxal_district'] == True:
                    key = str(state) + str(gender) + str(post) + "N" + str(category)
                    if key in vacancy_dict and vacancy_dict[key]['current'] > 0:
                        min_marks_prev = vacancy_dict[key].get('min_marks_prev', 0)
                        min_marks_parta_prev = vacancy_dict[key].get('min_marks_parta_prev', 0)
                        min_marks_partb_prev = vacancy_dict[key].get('min_marks_partb_prev', 0)
                        min_marks_cand_dob_prev = vacancy_dict[key].get('min_marks_cand_dob_prev', pd.Timestamp.min)
 
                        allocate = False
                        if total_marks > min_marks_prev:
                            allocate = True
                        elif total_marks == min_marks_prev:
                            if part_a > min_marks_parta_prev:
                                allocate = True
                            elif part_a == min_marks_parta_prev:
                                if part_b > min_marks_partb_prev:
                                    allocate = True
                                elif part_b == min_marks_partb_prev:
                                    if DOB < min_marks_cand_dob_prev:
                                        allocate = True
                                    elif DOB == min_marks_cand_dob_prev:
                                        allocate = True
 
                        if allocate:
                            post = vacancy_dict[key]['post_code']
                            allocated = update_allocation(candidates_df, state, gender, post, "N", category, cat1, allocated_against_ur, candidate, vacancy_dict)
                            if allocated:
                                vacancy_dict[key]['current'] -= 1
                                vacancy_dict[key]['allocated'] += 1
                                break
 
                # Border district logic
                if candidate['border_district'] == True:
                    key = str(state) + str(gender) + str(post) + "B" + str(category)
                    if key in vacancy_dict and vacancy_dict[key]['current'] > 0:
                        min_marks_prev = vacancy_dict[key].get('min_marks_prev', 0)
                        min_marks_parta_prev = vacancy_dict[key].get('min_marks_parta_prev', 0)
                        min_marks_partb_prev = vacancy_dict[key].get('min_marks_partb_prev', 0)
                        min_marks_cand_dob_prev = vacancy_dict[key].get('min_marks_cand_dob_prev', pd.Timestamp.min)
 
                        allocate = False
                        if total_marks > min_marks_prev:
                            allocate = True
                        elif total_marks == min_marks_prev:
                            if part_a > min_marks_parta_prev:
                                allocate = True
                            elif part_a == min_marks_parta_prev:
                                if part_b > min_marks_partb_prev:
                                    allocate = True
                                elif part_b == min_marks_partb_prev:
                                    if DOB < min_marks_cand_dob_prev:
                                        allocate = True
                                    elif DOB == min_marks_cand_dob_prev:
                                        allocate = True
 
                        if allocate:
                            post = vacancy_dict[key]['post_code']
                            allocated = update_allocation(candidates_df, state, gender, post, "B", category, cat1, allocated_against_ur, candidate, vacancy_dict)
                            if allocated:
                                vacancy_dict[key]['current'] -= 1
                                vacancy_dict[key]['allocated'] += 1
                                break
 
                # General allocation logic
                key = str(state) + str(gender) + str(post) + "G" + str(category)
                if key in vacancy_dict and vacancy_dict[key]['current'] > 0:
                    min_marks_prev = vacancy_dict[key].get('min_marks_prev', 0)
                    min_marks_parta_prev = vacancy_dict[key].get('min_marks_parta_prev', 0)
                    min_marks_partb_prev = vacancy_dict[key].get('min_marks_partb_prev', 0)
                    min_marks_cand_dob_prev = vacancy_dict[key].get('min_marks_cand_dob_prev', pd.Timestamp.min)
 
                    allocate = False
                    if total_marks > min_marks_prev:
                        allocate = True
                    elif total_marks == min_marks_prev:
                        if part_a > min_marks_parta_prev:
                            allocate = True
                        elif part_a == min_marks_parta_prev:
                            if part_b > min_marks_partb_prev:
                                allocate = True
                            elif part_b == min_marks_partb_prev:
                                if DOB < min_marks_cand_dob_prev:
                                    allocate = True
                                elif DOB == min_marks_cand_dob_prev:
                                    allocate = True
 
                    if allocate:
                        post = vacancy_dict[key]['post_code']
                        allocated = update_allocation(candidates_df, state, gender, post, "G", category, cat1, allocated_against_ur, candidate, vacancy_dict)
                        if allocated:
                            vacancy_dict[key]['current'] -= 1
                            vacancy_dict[key]['allocated'] += 1
                            break
 
            if allocated:
                break
 
    return vacancy_dict

In [245]:
def adjust_vacancy(candidates_df, vacancy_df):
    vacancy_df['current'] = vacancy_df['initial'] - vacancy_df['allocated_hc']
    vacancy_df['allocated'] = 0
    vacancy_df['left_vacancy'] = 0
        
    vacancy_df['allocated_hc_prev'] = vacancy_df['allocated_hc']
    vacancy_df['allocated_hc'] = 0

    return vacancy_df

#### EXECUTION

##### FIRST TIME

In [246]:
upd_vacancy_dict = allocate_candidates(cand, vacancy_dict)
updated_vacancy_df = pd.DataFrame.from_dict(upd_vacancy_dict, orient='index')

In [247]:
updated_vacancy_df['current'].sum()

8735

In [248]:
updated_vacancy_df['initial'].sum()

59068

In [249]:
cand[cand['allocated_category'].notnull()].shape[0]

50333

##### ADJUST LOOP

In [250]:
updated_vacancy_df['allocated'].sum()

50333

In [251]:
test_vac = updated_vacancy_df.copy()

In [252]:
updated_vacancy_df = test_vac.copy()

In [253]:
i=0
while (updated_vacancy_df['allocated_hc_prev'] != updated_vacancy_df['allocated_hc']).any():
    
    i+=1
    print('Adjust Attempt-'+str(i))
    
    upd_vacancy_df = adjust_vacancy(cand, updated_vacancy_df)
    upd_vacancy_df['left_vacancy'] = upd_vacancy_df['current']
    
    vacancy_dict = {}
    
    upd_vacancy_df['key'] = upd_vacancy_df['state_code'].astype(str) + upd_vacancy_df['gender'].astype(str) + upd_vacancy_df['post_code'].astype(str) + upd_vacancy_df['area'].astype(str) + upd_vacancy_df['category_code'].astype(str)
    
    for index, row in upd_vacancy_df.iterrows():
        key = row['key']
        vacancy_dict[key] = row.to_dict()
        vacancy_dict[key]['allocated_hc_prev'] = row['allocated_hc_prev']
    
    cand[['allocated_category','allocated_state', 'allocated_area', 'allocated_against_ur','allocated_post']] = None
    upd_vacancy_dict = allocate_candidates(cand, vacancy_dict)
    updated_vacancy_df = pd.DataFrame.from_dict(upd_vacancy_dict, orient='index')

Adjust Attempt-1


In [254]:
updated_vacancy_df['allocated_hc_prev'].sum()

287

In [255]:
updated_vacancy_df['allocated_hc'].sum()

287

#### WRITING TO CSV

In [256]:
updated_vacancy_df[['min_marks', 'min_marks_parta', 'min_marks_partb', 'min_marks_cand_dob', 'min_marks_merit']] = None

In [257]:
def find_lowest_marks(candidates_df, vacancy_df):
    candidates_df['key'] = (candidates_df['allocated_state'].astype(str) +
                            candidates_df['gender'].astype(str) +
                            candidates_df['allocated_post'].astype(str) +
                            candidates_df['allocated_area'].astype(str) +
                            candidates_df['allocated_category'].astype(str))

    vacancy_df['key'] = (vacancy_df['state_code'].astype(str) +
                         vacancy_df['gender'].astype(str) +
                         vacancy_df['post_code'].astype(str) +
                         vacancy_df['area'].astype(str) +
                         vacancy_df['category_code'].astype(str))

    highest_merit_candidates = candidates_df.loc[candidates_df.groupby('key')['merit'].idxmax()]

    vacancy_df_merged = vacancy_df.merge(highest_merit_candidates[['key', 'total_marks', 'parta_gi', 'partb_ga', 'dob', 'merit']],
                                  on='key', how='left')

    vacancy_df_merged['min_marks'] = vacancy_df_merged['total_marks']
    vacancy_df_merged['min_marks_parta'] = vacancy_df_merged['parta_gi']
    vacancy_df_merged['min_marks_partb'] = vacancy_df_merged['partb_ga']
    vacancy_df_merged['min_marks_cand_dob'] = vacancy_df_merged['dob']
    vacancy_df_merged['min_marks_merit'] = vacancy_df_merged['merit']

    return vacancy_df_merged

In [258]:
final_vacancy_df =  find_lowest_marks(cand, updated_vacancy_df)

In [259]:
final_vacancy_df[(final_vacancy_df['total_marks'].isnull())]['allocated'].value_counts()

allocated
0    785
Name: count, dtype: int64

In [260]:
final_vacancy_df['allocated'].sum()

50051

In [261]:
cand = cand.sort_values(by = "merit", ascending = True)

In [262]:
final_vacancy_df['left_vacancy'] = final_vacancy_df['current']
final_vacancy_df['current'] = final_vacancy_df['left_vacancy'] + final_vacancy_df['allocated']
 

In [264]:
cand.to_csv(r"allocated_candidates_ctgd2025.csv", index = False)
cand[cand['allocated_category'].notnull()].to_csv(r"only_allocated_candidates.csv", index = False)
final_vacancy_df.to_csv(r"allocated_vacancy_ctgd.csv", index = False)